# 📡 Telecom X — Previsão de Evasão de Clientes (Churn)
## Parte 2 — Pipeline de Machine Learning

**Objetivo:** Construir modelos preditivos para identificar clientes com maior probabilidade de cancelar seus serviços.

---
### Pipeline:
1. Carregamento e inspeção dos dados tratados
2. Remoção de colunas irrelevantes
3. Encoding de variáveis categóricas
4. Análise de proporção de evasão (balanceamento)
5. Normalização (para modelos sensíveis à escala)
6. Análise de correlação
7. Análises direcionadas
8. Criação de modelos
9. Avaliação dos modelos
10. Importância das variáveis + Conclusão estratégica

In [ ]:
# Instalação das dependências (se necessário)
# !pip install pandas numpy scikit-learn matplotlib seaborn imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve, auc, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

print('✅ Bibliotecas importadas com sucesso!')

---
## Etapa 1 — Carregamento dos Dados

In [ ]:
# Carregamento do arquivo tratado (Parte 1 do desafio)
df = pd.read_csv('data/telecom_tratado.csv')

print(f'Shape: {df.shape}')
print(f'Valores nulos: {df.isnull().sum().sum()}')
df.head()

In [ ]:
df.info()

---
## Etapa 2 — Remoção de Colunas Irrelevantes

In [ ]:
# Identificadores únicos (ex: id_cliente) não contribuem para a previsão
# e podem prejudicar o desempenho dos modelos.

# Exemplo: df = df.drop(columns=['id_cliente'])
# Neste dataset o ID já foi removido na Parte 1.

print(f'Colunas após limpeza: {df.shape[1]}')
print(f'Colunas: {df.columns.tolist()}')

---
## Etapa 3 — Encoding de Variáveis Categóricas

In [ ]:
categoricas = df.select_dtypes(include='object').columns.tolist()
print(f'Variáveis categóricas ({len(categoricas)}): {categoricas}')

# One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=categoricas, drop_first=False)

# Converter booleanos para int
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print(f'\nShape após encoding: {df_encoded.shape}')

---
## Etapa 4 — Proporção de Evasão e Balanceamento

In [ ]:
contagem = df['evasao'].value_counts()
proporcao = df['evasao'].value_counts(normalize=True) * 100

print('Distribuição da variável alvo:')
print(f'  Ativos:   {contagem[0]:,} ({proporcao[0]:.1f}%)')
print(f'  Evadidos: {contagem[1]:,} ({proporcao[1]:.1f}%)')
print(f'  Razão: 1:{contagem[0]/contagem[1]:.1f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].pie(contagem.values, labels=['Ativo', 'Evadido'],
            colors=['#2ECC71', '#E94560'], autopct='%1.1f%%', startangle=90)
axes[0].set_title('Proporção de Evasão')
axes[1].bar(['Ativo', 'Evadido'], contagem.values, color=['#2ECC71', '#E94560'])
axes[1].set_title('Contagem por Classe')
plt.tight_layout()
plt.show()

# Estratégia de balanceamento adotada: class_weight='balanced' nos modelos
# Alternativa: SMOTE (imbalanced-learn) para oversampling sintético

---
## Etapa 5 — Separação dos Dados e Normalização

In [ ]:
TARGET = 'evasao'
X = df_encoded.drop(columns=[TARGET])
y = df_encoded[TARGET]
feature_names = X.columns.tolist()

# Divisão 80/20 com estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Treino: {X_train.shape[0]:,} amostras')
print(f'Teste:  {X_test.shape[0]:,} amostras')

# StandardScaler — apenas para modelos sensíveis à escala (ex: Regressão Logística)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print('\n✅ Normalização aplicada para Regressão Logística')

---
## Etapa 6 — Análise de Correlação

In [ ]:
# Correlação com a variável alvo
corr_target = df_encoded.corr(numeric_only=True)['evasao'].drop('evasao')
top15 = corr_target.abs().nlargest(15)
top15_vals = corr_target[top15.index].sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cores = ['#E94560' if v > 0 else '#2ECC71' for v in top15_vals.values]
axes[0].barh(top15_vals.index, top15_vals.values, color=cores)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.7)
axes[0].set_title('Top 15 Variáveis — Correlação com Evasão')
axes[0].set_xlabel('Correlação de Pearson')

# Heatmap das variáveis numéricas
num_cols = ['tempo_contrato_meses', 'cobranca_mensal', 'total_gasto', 'evasao', 'idoso']
sns.heatmap(df_encoded[num_cols].corr(), annot=True, fmt='.2f',
            cmap='RdYlGn_r', center=0, ax=axes[1],
            linewidths=2, linecolor='white')
axes[1].set_title('Matriz de Correlação — Numéricas')
plt.tight_layout()
plt.show()

---
## Etapa 7 — Análises Direcionadas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Tempo de contrato × Evasão
data_tempo = [df[df['evasao']==0]['tempo_contrato_meses'],
              df[df['evasao']==1]['tempo_contrato_meses']]
bp = axes[0,0].boxplot(data_tempo, labels=['Ativo', 'Evadido'], patch_artist=True)
for patch, color in zip(bp['boxes'], ['#2ECC71', '#E94560']):
    patch.set_facecolor(color)
axes[0,0].set_title('Tempo de Contrato × Evasão')
axes[0,0].set_ylabel('Meses')

# Cobrança mensal × Evasão
data_cob = [df[df['evasao']==0]['cobranca_mensal'],
            df[df['evasao']==1]['cobranca_mensal']]
bp2 = axes[0,1].boxplot(data_cob, labels=['Ativo', 'Evadido'], patch_artist=True)
for patch, color in zip(bp2['boxes'], ['#2ECC71', '#E94560']):
    patch.set_facecolor(color)
axes[0,1].set_title('Cobrança Mensal × Evasão')
axes[0,1].set_ylabel('R$/mês')

# Tipo de contrato × Taxa de evasão
contrato_churn = df.groupby('tipo_contrato')['evasao'].mean() * 100
axes[1,0].bar(contrato_churn.index, contrato_churn.values,
               color=['#E94560', '#2ECC71', '#3498DB'])
axes[1,0].set_title('Taxa de Evasão por Tipo de Contrato')
axes[1,0].set_ylabel('Taxa de Evasão (%)')

# Internet × Taxa de evasão
internet_churn = df.groupby('servico_internet')['evasao'].mean() * 100
axes[1,1].bar(internet_churn.index, internet_churn.values,
               color=['#E94560', '#3498DB', '#2ECC71'])
axes[1,1].set_title('Taxa de Evasão por Tipo de Internet')
axes[1,1].set_ylabel('Taxa de Evasão (%)')

plt.suptitle('Análises Direcionadas — Fatores de Evasão', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Etapa 8 — Criação dos Modelos

In [ ]:
# ── Modelo 1: Regressão Logística ─────────────────────────────────────────────
# Justificativa: Modelo interpretável e rápido. Requer normalização.
# class_weight='balanced' compensa o desequilíbrio de classes.
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]
print('✅ Regressão Logística treinada')

# ── Modelo 2: Random Forest ────────────────────────────────────────────────────
# Justificativa: Ensemble robusto, NÃO requer normalização.
# Fornece importância de variáveis nativamente.
rf = RandomForestClassifier(n_estimators=200, max_depth=15,
                             min_samples_split=10, random_state=42,
                             class_weight='balanced', n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]
print('✅ Random Forest treinado')

# ── Modelo 3: Árvore de Decisão (Bônus) ───────────────────────────────────────
# Justificativa: Alta interpretabilidade, NÃO requer normalização.
# max_depth=8 para evitar overfitting.
dt = DecisionTreeClassifier(max_depth=8, min_samples_split=20,
                             random_state=42, class_weight='balanced')
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]
print('✅ Árvore de Decisão treinada')

---
## Etapa 9 — Avaliação dos Modelos

In [ ]:
def calcular_metricas(nome, y_true, y_pred, y_prob):
    return {
        'Modelo':   nome,
        'Acurácia': round(accuracy_score(y_true, y_pred) * 100, 2),
        'Precisão': round(precision_score(y_true, y_pred, zero_division=0) * 100, 2),
        'Recall':   round(recall_score(y_true, y_pred) * 100, 2),
        'F1-Score': round(f1_score(y_true, y_pred) * 100, 2),
        'AUC-ROC':  round(roc_auc_score(y_true, y_prob) * 100, 2),
    }

resultados = [
    calcular_metricas('Regressão Logística', y_test, y_pred_lr, y_prob_lr),
    calcular_metricas('Random Forest',       y_test, y_pred_rf, y_prob_rf),
    calcular_metricas('Árvore de Decisão',   y_test, y_pred_dt, y_prob_dt),
]
df_resultados = pd.DataFrame(resultados)
df_resultados.set_index('Modelo', inplace=True)
df_resultados.style.highlight_max(axis=0, color='#C6F6D5')

In [ ]:
# Matrizes de Confusão
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
modelos = ['Regressão Logística', 'Random Forest', 'Árvore de Decisão']
cores_mod = ['#E94560', '#1A1F36', '#3498DB']

for ax, y_pred, nome, cor in zip(axes,
    [y_pred_lr, y_pred_rf, y_pred_dt], modelos, cores_mod):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                cmap=sns.light_palette(cor, as_cmap=True),
                xticklabels=['Ativo', 'Evadido'],
                yticklabels=['Ativo', 'Evadido'],
                annot_kws={'size': 14, 'weight': 'bold'})
    ax.set_title(nome, fontweight='bold')
    ax.set_ylabel('Real'); ax.set_xlabel('Previsto')

plt.suptitle('Matrizes de Confusão', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Curvas ROC
fig, ax = plt.subplots(figsize=(8, 7))
for y_prob, nome, cor in zip([y_prob_lr, y_prob_rf, y_prob_dt], modelos, cores_mod):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=cor, lw=2, label=f'{nome} (AUC={auc_val:.3f})')
ax.plot([0,1],[0,1], 'k--', alpha=0.5)
ax.set_xlabel('Taxa de Falsos Positivos')
ax.set_ylabel('Taxa de Verdadeiros Positivos')
ax.set_title('Curvas ROC', fontsize=13, fontweight='bold')
ax.legend()
plt.show()

---
## Etapa 10 — Importância das Variáveis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Random Forest
imp_rf = pd.Series(rf.feature_importances_, index=feature_names).nlargest(20).sort_values()
axes[0].barh(imp_rf.index, imp_rf.values, color='#1A1F36', alpha=0.85)
axes[0].set_title('Random Forest — Top 20 Variáveis', fontweight='bold')
axes[0].set_xlabel('Importância (Gini)')

# Regressão Logística
coef_lr = pd.Series(lr.coef_[0], index=feature_names)
top20_lr = coef_lr.abs().nlargest(20)
top20_lr_vals = coef_lr[top20_lr.index].sort_values()
cores_lr = ['#E94560' if v > 0 else '#2ECC71' for v in top20_lr_vals.values]
axes[1].barh(top20_lr_vals.index, top20_lr_vals.values, color=cores_lr)
axes[1].axvline(0, color='gray', linestyle='--')
axes[1].set_title('Regressão Logística — Coeficientes', fontweight='bold')
axes[1].set_xlabel('Coeficiente')

plt.tight_layout()
plt.show()

---
## Conclusão Estratégica

### 🔍 Principais Fatores de Evasão

| Fator | Impacto | Ação Recomendada |
|---|---|---|
| **Contrato Mês a Mês** | 🔴 Alto (≈3x mais evasão) | Incentivar migração para contratos anuais com descontos |
| **Internet Fibra Óptica** | 🔴 Alto | Revisar qualidade/preço do serviço |
| **Cobrança Mensal Alta** | 🟠 Médio | Criar planos mais acessíveis |
| **Cheque Eletrônico** | 🟠 Médio | Incentivar débito automático com benefício |
| **Tempo de Contrato < 12 meses** | 🔴 Alto | Programa de boas-vindas e fidelização nos primeiros meses |
| **Contrato 2 Anos** | 🟢 Protetor | Expandir oferta de contratos longos |
| **Suporte Técnico incluído** | 🟢 Protetor | Incluir suporte nos planos básicos |

### 🤖 Modelo Recomendado
**Regressão Logística** — Melhor Recall (71,24%) e F1-Score (58,27%), com alta interpretabilidade.

### 🚀 Próximos Passos
- Tuning de hiperparâmetros com GridSearchCV
- Testar XGBoost / LightGBM
- Aplicar SMOTE para balanceamento de classes
- Deploy via Flask/FastAPI
- Dashboard de monitoramento em tempo real